In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import json

with open("data/documents.json", 'rt') as f_in:
    raw_data = json.load(f_in)

documents = []

for course_list in raw_data:
  for doc in course_list['documents']:
    doc['course'] = course_list['course']
    documents.append(doc)



In [ ]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch('http://localhost:9200')
es_client.info()



In [ ]:
index_settings = {
        "settings": {
            "number_of_shards": 1,
            "number_of_replicas": 0
        },
        "mappings": {
            "properties": {
                "text": {"type": "text"},
                "section": {"type": "text"},
                "question": {"type": "text"},
                "course": {"type": "keyword"} 
            }
        }
    }
    
index_name = "course_questions"
    
es_client.indices.create(index=index_name, body=index_settings)

for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

In [ ]:
from tqdm.auto import tqdm 

def elasticSearch(query):
    
   

    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)

    search_res = []

    for doc in response['hits']['hits']:
        search_res.append(doc['_source'])

    return search_res


In [ ]:
def RAG(query):
    response = elasticSearch(query)
    prompt = genPrompt(response, query)
    res = mistralLLM(prompt)

    return res

In [ ]:
def genPrompt(search_result, query):
    prompt_template = """
You are an course assistant that guides students regarding courses. 
You will answer based on the provided context pnly and If the answer is not in the context, say:
"I don't know based on the provided course materials."

QUESTION: {question}

CONTEXT: {context}
"""

    context = ""
    
    for doc in search_result:
        context = context + f"\n question:{doc['question']}\n section:{doc['section']}\n answer:{doc['text']}\n\n"
    
    
    prompt = prompt_template.format(
        question=query,
        context=context
    )
    
    return prompt

In [ ]:
import os
from mistralai import Mistral

apiKey = os.getenv("MISTRAL_API_KEY")

def mistralLLM(prompt):
    client = Mistral(api_key=apiKey)

    inputs = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    completion_args = {
        "temperature": 0.3,
        "max_tokens": 2048,
        "top_p": 1
    }

    tools = []

    response = client.beta.conversations.start(
        inputs=inputs,
        model="mistral-medium-latest",
        instructions="""""",
        completion_args=completion_args,
        tools=tools,
    )
    return response.outputs[0].content

In [ ]:

print(RAG("how to run a car"))